# 01 — Data Cleaning

Takes the raw CFPB file and turns it into clean train/test files the models can use.

Steps:
1. Load the raw CSV
2. Tidy the column names
3. Keep only complaints that have a written narrative
4. Build the "escalation" labels
5. Remove duplicate (copy-paste) complaints
6. Split into train (older) and test (newer) by date
7. Balance the TRAIN set so rare products are not drowned out (TEST left as-is)
8. Save the results


In [1]:
import os
import pandas as pd

# Work from the project root so the "data/..." paths are correct.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# We now use the 6-month file (Jan-Jun 2023) instead of the single March file.
# More data, and especially more examples of the RARE product classes, is the
# main reason we switched. If you ever want the old March run back, point this
# at "data/raw/cfpb_complaints_2023-03.csv" and re-run.
RAW_FILE = "data/raw/cfpb_2023-01-01_to_2023-06-30.csv"


In [2]:
# 1) Load the raw file.
df = pd.read_csv(RAW_FILE, low_memory=False)
print("Loaded", len(df), "rows")

Loaded 234719 rows


In [3]:
# 2) Tidy the column names: lowercase, and turn spaces and dashes into underscores.
#    Example: "Consumer complaint narrative" -> "consumer_complaint_narrative"
df.columns = (df.columns
              .str.lower()
              .str.replace("?", "", regex=False)
              .str.replace("-", "_", regex=False)
              .str.replace(" ", "_", regex=False))
print(list(df.columns))

['date_received', 'product', 'sub_product', 'issue', 'sub_issue', 'consumer_complaint_narrative', 'company_public_response', 'company', 'state', 'zip_code', 'tags', 'submitted_via', 'date_sent_to_company', 'company_response_to_consumer', 'timely_response', 'complaint_id']


In [4]:
# 3) Keep only complaints that include a written narrative (that's what the models read).
df = df.dropna(subset=["consumer_complaint_narrative"])
print(len(df), "rows have a narrative")

# Drop complaints still "In progress" — no final outcome yet, so we can't label them.
df = df[df["company_response_to_consumer"] != "In progress"]

234719 rows have a narrative


In [5]:
# 4) Build the labels.
#    We flag two DIFFERENT problems separately, plus a combined flag:
#      - the company had to pay the customer      -> is_monetary_relief
#      - the company missed its response deadline -> is_untimely
df["is_monetary_relief"] = (df["company_response_to_consumer"] == "Closed with monetary relief").astype(int)
df["is_untimely"] = ((df["timely_response"] == "No") |
                     (df["company_response_to_consumer"] == "Untimely response")).astype(int)
df["is_escalated"] = ((df["is_monetary_relief"] == 1) | (df["is_untimely"] == 1)).astype(int)

print("monetary relief:", df["is_monetary_relief"].sum())
print("untimely:       ", df["is_untimely"].sum())
print("escalated:      ", df["is_escalated"].sum())

monetary relief: 5751
untimely:        1378
escalated:       7038


In [6]:
# 5) Remove duplicate complaints (credit-repair firms send the same template for many people).
#    Two narratives count as the same if they match after lowercasing and squeezing spaces.
df["date_received"] = pd.to_datetime(df["date_received"])
df = df.sort_values("date_received")      # oldest first, so we keep the earliest copy

df["narrative_key"] = (df["consumer_complaint_narrative"]
                       .str.lower()
                       .str.replace(r"\s+", " ", regex=True)   # squeeze runs of spaces into one
                       .str.strip())

before = len(df)
df = df.drop_duplicates(subset="narrative_key")
print("Removed", before - len(df), "duplicates;", len(df), "rows left")

Removed 61251 duplicates; 173468 rows left


In [7]:
# 6) Split into train and test BY DATE (not randomly).
#    Train on older complaints, test on the newest 20%. This copies real life:
#    you always predict on complaints that arrive AFTER the ones you learned from.
split_point = int(len(df) * 0.8)
train = df.iloc[:split_point]
test = df.iloc[split_point:]
print("train:", len(train), " test:", len(test))

train: 138774  test: 34694


In [8]:
# 7) Balance the TRAINING set so the rare products are not drowned out.
#    Credit reporting is ~65% of complaints; without this, the models barely
#    see payday loans, vehicle loans, money transfers, etc. We randomly cap
#    each big product at MAX_PER_CLASS rows, and keep ALL rows of the smaller
#    products. This is a soft cap, not perfect balance.
#
#    IMPORTANT: we balance TRAIN only. The TEST set is left at its natural,
#    real-world class mix, so the scores we report later are honest.
#
#    We ALSO keep a copy of the natural (unbalanced) train. Notebook 04 uses it
#    to build an honest validation set from the real class mix.
train_natural = train.copy()

MAX_PER_CLASS = 8000

balanced_parts = []
for product_name in train["product"].unique():
    rows = train[train["product"] == product_name]
    if len(rows) > MAX_PER_CLASS:
        rows = rows.sample(n=MAX_PER_CLASS, random_state=42)   # random cap for big classes
    balanced_parts.append(rows)

# Put the pieces back together and re-sort oldest -> newest.
train = pd.concat(balanced_parts).sort_values("date_received").reset_index(drop=True)

print("balanced train:", len(train), "rows")
print("natural train kept for nb04 val:", len(train_natural), "rows")
print(train["product"].value_counts())


balanced train: 47121 rows
natural train kept for nb04 val: 138774 rows
product
Credit card or prepaid card                                                     8000
Debt collection                                                                 8000
Checking or savings account                                                     8000
Credit reporting, credit repair services, or other personal consumer reports    8000
Mortgage                                                                        5736
Money transfer, virtual currency, or money service                              3254
Vehicle loan or lease                                                           3156
Payday loan, title loan, or personal loan                                       1663
Student loan                                                                    1312
Name: count, dtype: int64


In [9]:
# 8) Save. Keep only the columns the models are allowed to use.
#    IMPORTANT: we drop "company_response_to_consumer" and "timely_response".
#    They tell you how the complaint ENDED, which you would not know in real life,
#    so letting the model see them would be cheating. We only needed them for the labels.
columns_to_keep = [
    "complaint_id", "date_received", "product", "sub_product", "issue", "sub_issue",
    "company", "state", "consumer_complaint_narrative",
    "is_monetary_relief", "is_untimely", "is_escalated",
]

os.makedirs("data/processed", exist_ok=True)
train[columns_to_keep].to_parquet("data/processed/train.parquet", index=False)
# The natural (unbalanced) train, used by notebook 04 to carve an honest
# validation set. The baseline (nb03) still uses the balanced train.parquet above.
train_natural[columns_to_keep].to_parquet("data/processed/train_natural.parquet", index=False)
test[columns_to_keep].to_parquet("data/processed/test.parquet", index=False)
print("Saved train.parquet, train_natural.parquet, and test.parquet")

Saved train.parquet, train_natural.parquet, and test.parquet
